# Fraud / Anomaly Detection
## Notebook 12 — Model Comparison

### Objective
Create an AutoML-style leaderboard from saved model score files.

## Load score artifacts

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
files = {'Isolation Forest':'isolation_forest_scores.csv', 'LOF':'lof_scores.csv', 'One-Class SVM':'one_class_svm_scores.csv', 'DBSCAN':'dbscan_scores.csv'}
missing = [f for f in files.values() if not (ROOT/'reports'/f).exists()]
if missing: raise FileNotFoundError('Run notebooks 07–10 first. Missing: ' + ', '.join(missing))

**Interpretation.** Each score artifact includes its own sampled labels, so metrics remain honest for that model's working set.

**ML decision.** Rank primarily by PR-AUC due to severe class imbalance.

## Build leaderboard

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from src.fraud_detection import evaluate_scores
rows=[]
for model, filename in files.items():
    frame=pd.read_csv(ROOT/'reports'/filename)
    rows.append(evaluate_scores(frame.label, frame.score, model))
leaderboard=pd.DataFrame(rows).sort_values(['pr_auc','recall'], ascending=False).reset_index(drop=True)
leaderboard.to_csv(ROOT/'reports/model_leaderboard.csv', index=False)
leaderboard

**Interpretation.** PR-AUC measures ranking performance under imbalance; threshold metrics use a predicted anomaly count equal to observed fraud prevalence for comparison.

**ML decision.** Advance the top-ranked deployable model to final evaluation.